# 2 · Transcribe, classify, and review

Use a **fresh Colab GPU runtime** for this notebook. Install once, choose **Runtime → Restart session**, and continue at configuration, skipping installation. After restarting, run cells from configuration onward; running all cells reinstalls packages and requires another restart. The import cell checks the loaded and installed NumPy versions. Upload the extraction ZIP from notebook 1. You can also upload this notebook's previous analysis ZIP to resume; compatible per-clip predictions are reused.

[faster-whisper](https://github.com/SYSTRAN/faster-whisper) transcribes the spoken language without translation. [emotion2vec+ base](https://huggingface.co/emotion2vec/emotion2vec_plus_base) estimates angry, disgusted, fearful, happy, neutral, other, sad, surprised, and unknown from audio. Its scores are model outputs, not calibrated probabilities for anime; Japanese acting, music, and exaggerated delivery require validation.

**Kindness, warmth, naturalness, worry, and stress are separate listening annotations.** This model has no validated outputs for them. Neutral emotion does not mean natural acting; fear does not automatically mean worry or stress. The optional review player lets you assign these attributes and correct transcripts. No external LLM service is used.

Definitions for this exploratory dataset: kindness = compassionate/supportive delivery; warmth = affectionate/soothing vocal tone; naturalness = conversational delivery rather than exaggerated acting; worry = audible apprehension; stress = audible tension/pressure. They may coexist. Use `uncertain/unrated` when you cannot decide, and document changed definitions in `ATTRIBUTE_DEFINITIONS` before comparing runs.

All outputs are private research data. Download progress before ending the runtime; Colab storage is temporary. Clear notebook outputs and personal inputs before publishing code.

In [ ]:
# Run ONLY in Colab. Restart afterward, then skip this cell.
import subprocess
import sys

if "google.colab" not in sys.modules:
    raise RuntimeError("Open this notebook in Google Colab before installing dependencies.")

required_packages: list[str] = [
    "numpy==2.2.6",
    "torch==2.8.0",
    "torchvision==0.23.0",
    "torchaudio==2.8.0",
    "faster-whisper==1.2.1",
    "ctranslate2==4.8.2",
    "funasr==1.4.16",
    "soundfile==0.13.1",
    "ipywidgets>=8,<9",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", *required_packages],
    check=True,
)

# Refresh NumPy's Python files and compiled extensions together, even if already installed.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir",
     "--force-reinstall", "--no-deps", "numpy==2.2.6"],
    check=True,
)
_VOICE_STAT_RESTART_REQUIRED: bool = True
print("Setup complete. Choose Runtime → Restart session, then continue at configuration.")

In [ ]:
INPUT_ZIP: str = ""  # Empty opens the upload dialog.
LANGUAGE_OVERRIDE: str | None = None  # None uses the extraction's language; set e.g. "ja" or "en".
ASR_MODEL: str = "Systran/faster-whisper-small"
ASR_REVISION: str | None = "536b0662742c02347bc0e980a01041f333bce120"
# For a larger multilingual model, change ASR_MODEL and set ASR_REVISION = None.
ASR_COMPUTE_TYPE: str = "int8_float16"
ASR_BEAM_SIZE: int = 5
ASR_WORKERS: int = 2  # Independent clips in parallel; use 1 for minimum GPU memory.
EMOTION_MODEL: str = "emotion2vec/emotion2vec_plus_base"
EMOTION_REVISION: str = "b318240bfe67db81a8c572ecb37ce9c3759b81c9"
EMOTION_WINDOW_SECONDS: float = 15.0
MIN_EMOTION_SPEECH_SECONDS: float = 0.5
EMOTION_ACCEPT_SCORE: float = 0.60
ASR_LOW_LOGPROB: float = -1.0
MAX_CLIPS_THIS_SESSION: int | None = None  # e.g. 50 to limit GPU use; rerun to process the next batch.
ATTRIBUTE_DEFINITIONS: dict[str, str] = {
    "kindness": "Compassionate or supportive vocal delivery.",
    "warmth": "Affectionate or soothing vocal tone.",
    "naturalness": "Conversational delivery rather than exaggerated acting.",
    "worry": "Audible apprehension.",
    "stress": "Audible tension or pressure.",
}

In [ ]:
if globals().get("_VOICE_STAT_RESTART_REQUIRED", False):
    raise RuntimeError("Restart the Colab session after installation, then skip setup and continue at configuration.")

import gc
import hashlib
import importlib.metadata
import json
import math
import os
import shutil
import uuid
import zipfile
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, TypeAlias
from time import perf_counter

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

import ipywidgets as widgets
import numpy as np

installed_numpy_version: str = importlib.metadata.version("numpy")
if np.__version__ != installed_numpy_version or installed_numpy_version != "2.2.6":
    raise RuntimeError(
        f"NumPy version mismatch: loaded {np.__version__}, installed {installed_numpy_version}; "
        "this notebook requires 2.2.6. Run setup, choose Runtime → Restart session, "
        "then continue at configuration, skipping setup."
    )

from numpy.typing import NDArray
import pandas as pd
import soundfile as sf
import torch
from google.colab import files, output
from huggingface_hub import HfApi, snapshot_download
from IPython.display import Audio, clear_output, display
from tqdm.auto import tqdm

JsonObject: TypeAlias = dict[str, Any]
FloatArray: TypeAlias = NDArray[np.float32]

if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab GPU runtime and restart the session.")
if EMOTION_WINDOW_SECONDS <= 0 or MIN_EMOTION_SPEECH_SECONDS < 0:
    raise ValueError("Emotion window must be positive and minimum speech duration nonnegative.")
if not 0 <= EMOTION_ACCEPT_SCORE <= 1:
    raise ValueError("EMOTION_ACCEPT_SCORE must be between zero and one.")
if MAX_CLIPS_THIS_SESSION is not None and MAX_CLIPS_THIS_SESSION < 1:
    raise ValueError("MAX_CLIPS_THIS_SESSION must be positive or None.")
if ASR_WORKERS < 1:
    raise ValueError("ASR_WORKERS must be positive.")
output.enable_custom_widget_manager()


def write_json(path: Path, value: JsonObject) -> None:
    # Replace only after the new JSON is complete, so interrupted inference remains resumable.
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    temporary.replace(path)


def signature(value: JsonObject) -> str:
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def extract_zip_safely(archive_path: Path, destination: Path) -> None:
    destination = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError(f"Archive member escapes its directory: {member.filename}")
            if (member.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("Archive symlinks are not supported.")
        archive.extractall(destination)


if not INPUT_ZIP:
    uploaded = files.upload()
    names: list[str] = [name for name in uploaded if Path(name).suffix.lower() == ".zip"]
    if len(names) != 1:
        raise ValueError("Upload one extraction or analysis ZIP.")
    INPUT_ZIP: str = names[0]
    del uploaded
work_root: Path = Path("/content/voice-stat-analysis") / uuid.uuid4().hex[:12]
work_root.mkdir(parents=True)
extract_zip_safely(Path(INPUT_ZIP), work_root)
manifest_paths: list[Path] = list(work_root.rglob("manifest.json"))
if len(manifest_paths) != 1:
    raise ValueError("Expected exactly one manifest.json in this archive.")
bundle_root: Path = manifest_paths[0].parent
manifest: JsonObject = json.loads(manifest_paths[0].read_text(encoding="utf-8"))
if manifest.get("schema_version") != 1 or manifest.get("kind") not in {"voice_stat_extraction", "voice_stat_analysis"}:
    raise ValueError("Unsupported archive schema.")
records: list[JsonObject] = json.loads((bundle_root / "segments.json").read_text(encoding="utf-8"))["segments"]
if not records:
    raise ValueError("The archive contains no segments.")
for record in records:
    clip_path = (bundle_root / record["audio_file"]).resolve()
    if not clip_path.is_relative_to(bundle_root.resolve()):
        raise ValueError("Segment path escapes the bundle.")
    if file_sha256(clip_path) != record["audio_sha256"]:
        raise ValueError(f"Audio checksum mismatch for {record['observation_id']}.")

asr_directory: Path = bundle_root / "transcripts"
emotion_directory: Path = bundle_root / "emotions"
review_directory: Path = bundle_root / "annotations"
for directory in [asr_directory, emotion_directory, review_directory]:
    directory.mkdir(exist_ok=True)
LANGUAGE: str = LANGUAGE_OVERRIDE or manifest["language"]
if ASR_REVISION is None:
    ASR_REVISION: str | None = HfApi().model_info(ASR_MODEL).sha
asr_settings: JsonObject = {"model": ASR_MODEL, "revision": ASR_REVISION, "language": LANGUAGE,
                "compute_type": ASR_COMPUTE_TYPE, "beam_size": ASR_BEAM_SIZE,
                "condition_on_previous_text": False, "vad_filter": True,
                "word_timestamps": True, "version": importlib.metadata.version("faster-whisper")}
emotion_settings: JsonObject = {"model": EMOTION_MODEL, "revision": EMOTION_REVISION,
                    "window_seconds": EMOTION_WINDOW_SECONDS,
                    "minimum_speech_seconds": MIN_EMOTION_SPEECH_SECONDS,
                    "version": importlib.metadata.version("funasr")}


def cached_result(directory: Path, record: JsonObject, settings: JsonObject) -> JsonObject | None:
    path = directory / f"{record['observation_id']}.json"
    if not path.exists():
        return None
    value: JsonObject = json.loads(path.read_text(encoding="utf-8"))
    expected = signature({"audio_sha256": record["audio_sha256"], "settings": settings})
    return value if value.get("signature") == expected else None


# Choose one batch for both models. Completed clips are skipped on the next batch.
pending: list[JsonObject] = [record for record in records if cached_result(asr_directory, record, asr_settings) is None
           or cached_result(emotion_directory, record, emotion_settings) is None]
batch_records: list[JsonObject] = pending if MAX_CLIPS_THIS_SESSION is None else pending[:MAX_CLIPS_THIS_SESSION]
print(f"{len(records)} total clips; {len(pending)} pending; {len(batch_records)} in this batch.")

## Transcribe

One model is loaded at a time to conserve GPU memory. Text, word times, and recognition diagnostics are stored per clip. Both clip-relative and episode-absolute timestamps are retained. Silence and music can produce hallucinated text; inspect low-quality results in review.

`ASR_WORKERS` controls concurrent clips using [faster-whisper's supported worker API](https://github.com/SYSTRAN/faster-whisper/blob/v1.2.1/faster_whisper/transcribe.py). Both the Python executor and the model use this count. Start with 2; use 1 if memory is tight, or try 4 and compare elapsed time. Each completed clip is saved independently, even if another worker is still running. Emotion analysis follows after these workers finish and remains sequential.

In [ ]:
from faster_whisper import WhisperModel

def transcribe_clip(record: JsonObject, model: WhisperModel) -> str:
    generated_segments, info = model.transcribe(
        str(bundle_root / record["audio_file"]), language=LANGUAGE,
        task="transcribe", beam_size=ASR_BEAM_SIZE, condition_on_previous_text=False,
        vad_filter=True, word_timestamps=True,
    )
    transcript_segments: list[JsonObject] = []
    offset = record["start_seconds"]
    for segment in generated_segments:
        transcript_segments.append({
            "clip_start_seconds": segment.start, "clip_end_seconds": segment.end,
            "episode_start_seconds": offset + segment.start,
            "episode_end_seconds": offset + segment.end,
            "text": segment.text.strip(), "avg_logprob": segment.avg_logprob,
            "no_speech_probability": segment.no_speech_prob,
            "words": [{"text": word.word, "clip_start_seconds": word.start,
                       "clip_end_seconds": word.end, "episode_start_seconds": offset + word.start,
                       "episode_end_seconds": offset + word.end, "probability": word.probability}
                      for word in segment.words or []],
        })
    text: str = " ".join(segment["text"] for segment in transcript_segments).strip()
    result: JsonObject = {
        "observation_id": record["observation_id"],
        "signature": signature({"audio_sha256": record["audio_sha256"], "settings": asr_settings}),
        "settings": asr_settings, "language": info.language,
        "language_probability": info.language_probability,
        "text": text, "segments": transcript_segments,
    }
    write_json(asr_directory / f"{record['observation_id']}.json", result)
    (asr_directory / f"{record['observation_id']}.txt").write_text(text, encoding="utf-8")
    return record["observation_id"]


transcription_started: float = perf_counter()
asr_pending: list[JsonObject] = [record for record in batch_records if cached_result(asr_directory, record, asr_settings) is None]
if asr_pending:
    model_path = snapshot_download(ASR_MODEL, revision=ASR_REVISION)
    asr_model: WhisperModel = WhisperModel(
        model_path, device="cuda", compute_type=ASR_COMPUTE_TYPE, num_workers=ASR_WORKERS,
    )
    try:
        with ThreadPoolExecutor(max_workers=ASR_WORKERS) as executor:
            completed = executor.map(transcribe_clip, asr_pending, [asr_model] * len(asr_pending))
            for observation_id in tqdm(completed, total=len(asr_pending), desc="Transcribe clips"):
                pass  # Workers save completed clips; consume results to surface errors.
    finally:
        del asr_model
        gc.collect()
        torch.cuda.empty_cache()
print(f"Transcription phase: {perf_counter() - transcription_started:.1f}s with {ASR_WORKERS} workers "
      "(including model download/load when needed).")

## Estimate broad acoustic emotions

Long clips are analyzed in bounded windows. Each window's scores and times are saved, then averaged with weights based on detected target speech duration. This is a clip summary; it can conceal an emotion change within the clip. Scores below `EMOTION_ACCEPT_SCORE`, model `unknown/other`, and insufficient speech remain unknown in the database summaries.

In [ ]:
from funasr import AutoModel


def target_speech_seconds(record: JsonObject, start: float, end: float) -> float:
    return sum(max(0.0, min(end, right) - max(start, left)) for left, right in record["speech_intervals"])


def canonical_emotion(label: str) -> str:
    # The model exposes bilingual names such as 生气/angry and <unk>.
    name = label.rsplit("/", 1)[-1].lower()
    aliases: dict[str, str] = {"<unk>": "unknown", "unk": "unknown"}
    name = aliases.get(name, name)
    allowed: set[str] = {"angry", "disgusted", "fearful", "happy", "neutral", "other", "sad", "surprised", "unknown"}
    if name not in allowed:
        raise ValueError(f"Unexpected model emotion label: {label}")
    return name


emotion_started: float = perf_counter()
emotion_pending: list[JsonObject] = [record for record in batch_records if cached_result(emotion_directory, record, emotion_settings) is None]
if emotion_pending:
    emotion_path = snapshot_download(EMOTION_MODEL, revision=EMOTION_REVISION)
    emotion_model: AutoModel = AutoModel(
        model=emotion_path, hub="hf", device="cuda:0", disable_update=True,
        trust_remote_code=False,
    )
    for record in tqdm(emotion_pending, desc="Classify acoustic emotion"):
        clip: FloatArray
        rate: int
        clip, rate = sf.read(bundle_root / record["audio_file"], dtype="float32")
        if rate != 16000 or clip.ndim != 1:
            raise ValueError("Emotion input must be the 16 kHz mono extraction audio.")
        windows: list[JsonObject] = []
        if record["speech_seconds"] >= MIN_EMOTION_SPEECH_SECONDS:
            count = max(1, math.ceil((len(clip) / rate) / EMOTION_WINDOW_SECONDS))
            sample_edges = np.linspace(0, len(clip), count + 1, dtype=int)
            for start_sample, end_sample in zip(sample_edges[:-1], sample_edges[1:]):
                episode_start = record["start_seconds"] + start_sample / rate
                episode_end = record["start_seconds"] + end_sample / rate
                weight = target_speech_seconds(record, episode_start, episode_end)
                if weight < MIN_EMOTION_SPEECH_SECONDS:
                    continue
                prediction: JsonObject = emotion_model.generate(
                    input=clip[start_sample:end_sample], fs=rate,
                    granularity="utterance", extract_embedding=False,
                )[0]
                scores: dict[str, float] = {canonical_emotion(label): float(score)
                          for label, score in zip(prediction["labels"], prediction["scores"])}
                if not scores or not all(math.isfinite(score) for score in scores.values()):
                    raise ValueError("Emotion model returned invalid scores.")
                windows.append({"episode_start_seconds": episode_start,
                                "episode_end_seconds": episode_end,
                                "target_speech_seconds": weight, "scores": scores})
        scores: dict[str, float] = {}
        if windows:
            total_weight = sum(window["target_speech_seconds"] for window in windows)
            scores: dict[str, float] = {label: sum(window["scores"][label] * window["target_speech_seconds"] for window in windows) / total_weight
                      for label in windows[0]["scores"]}
        raw_label = max(scores, key=scores.get) if scores else "unknown"
        result: JsonObject = {
            "observation_id": record["observation_id"],
            "signature": signature({"audio_sha256": record["audio_sha256"], "settings": emotion_settings}),
            "settings": emotion_settings, "raw_label": raw_label,
            "top_score": scores.get(raw_label, 0.0), "scores": scores, "windows": windows,
        }
        write_json(emotion_directory / f"{record['observation_id']}.json", result)
    del emotion_model
    gc.collect()
    torch.cuda.empty_cache()
print(f"Emotion phase: {perf_counter() - emotion_started:.1f}s "
      "(including model download/load when needed).")

## Review any clip

Choose a clip, listen, check its identity, correct text if useful, and click **Save review** before switching clips. An accepted identity means you verified the clip belongs to this target; rejecting a clip excludes it from statistics. Missing custom ratings are unknown, never zero. A manual emotion label remains separate from the automatic prediction. Select `mixed` when no single emotion represents the clip.

Listening review is optional, but unreviewed results stay marked as estimates. Export after each GPU batch even if you defer review.

In [ ]:
EMOTION_CHOICES: list[str] = ["unrated", "angry", "disgusted", "fearful", "happy", "neutral", "sad", "surprised", "other", "unknown", "mixed"]


def load_review(record: JsonObject) -> JsonObject:
    path = review_directory / f"{record['observation_id']}.json"
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    return {"identity_status": "unreviewed", "emotion": None, "corrected_text": None,
            "attributes": {name: None for name in ATTRIBUTE_DEFINITIONS}, "notes": ""}


selector: widgets.Dropdown = widgets.Dropdown(
    options=[(f"{index + 1}: {record['start_seconds']:.3f}–{record['end_seconds']:.3f}s", index)
             for index, record in enumerate(records)], description="Clip:",
    layout=widgets.Layout(width="650px"),
)
review_display: widgets.Output = widgets.Output()


def show_review(change: JsonObject | None = None) -> None:
    record = records[selector.value]
    review: JsonObject = load_review(record)
    transcript = cached_result(asr_directory, record, asr_settings)
    emotion = cached_result(emotion_directory, record, emotion_settings)
    with review_display:
        clear_output(wait=True)
        print(record["observation_id"], "Extraction flags:", record["quality_flags"])
        print("Acoustic prediction:", None if emotion is None else emotion["scores"])
        display(Audio(filename=str(bundle_root / record["audio_file"])))
        identity: widgets.Dropdown = widgets.Dropdown(options=["unreviewed", "accepted", "rejected"], value=review["identity_status"], description="Identity:")
        emotion_choice: widgets.Dropdown = widgets.Dropdown(options=EMOTION_CHOICES, value=review["emotion"] or "unrated", description="Emotion:")
        original_text = "" if transcript is None else transcript["text"]
        text: widgets.Textarea = widgets.Textarea(value=review["corrected_text"] if review["corrected_text"] is not None else original_text,
                                description="Text:", layout=widgets.Layout(width="95%", height="100px"))
        ratings: dict[str, widgets.Dropdown] = {name: widgets.Dropdown(options=[("uncertain/unrated", None), ("absent", 0), ("present", 1)],
                                         value=review["attributes"].get(name), description=f"{name}:")
                   for name in ATTRIBUTE_DEFINITIONS}
        notes: widgets.Textarea = widgets.Textarea(value=review["notes"], description="Notes:", layout=widgets.Layout(width="95%"))
        save_button: widgets.Button = widgets.Button(description="Save review", button_style="success")
        feedback: widgets.Output = widgets.Output()

        def save_review(button: widgets.Button) -> None:
            value: JsonObject = {"observation_id": record["observation_id"], "identity_status": identity.value,
                     "emotion": None if emotion_choice.value == "unrated" else emotion_choice.value,
                     "corrected_text": text.value if text.value != original_text else None,
                     "attributes": {name: widget.value for name, widget in ratings.items()},
                     "attribute_definitions": ATTRIBUTE_DEFINITIONS,
                     "notes": notes.value, "reviewed_utc": datetime.now(timezone.utc).isoformat()}
            write_json(review_directory / f"{record['observation_id']}.json", value)
            with feedback:
                clear_output(wait=True)
                print("Saved. Run export below to include this review in your downloaded ZIP.")

        save_button.on_click(save_review)
        display(identity, emotion_choice, text, *ratings.values(), notes, save_button, feedback)


selector.observe(show_review, names="value")
display(selector, review_display)
show_review()

## Export progress and analysis

Rerun after saving reviews or processing another batch. The ZIP contains audio, original model predictions, per-clip plain-text transcripts, review JSON, and flattened `analysis.jsonl`/`analysis.csv` for notebook 3. You can re-upload this ZIP to resume on another day. Within this runtime, rerun the batch-selection cell, transcription, emotion, and export to process the next batch.

In [ ]:
analysis_rows: list[JsonObject] = []
for record in records:
    transcript = cached_result(asr_directory, record, asr_settings)
    emotion = cached_result(emotion_directory, record, emotion_settings)
    review: JsonObject = load_review(record)
    flags: list[str] = list(record["quality_flags"])
    if transcript is None:
        flags.append("transcription_pending")
    elif not transcript["text"]:
        flags.append("empty_transcript")
    elif any(segment["avg_logprob"] < ASR_LOW_LOGPROB or segment["no_speech_probability"] > 0.6 for segment in transcript["segments"]):
        flags.append("asr_needs_review")
    if emotion is None:
        flags.append("emotion_pending")
    raw_label = "unknown" if emotion is None else emotion["raw_label"]
    score = 0.0 if emotion is None else emotion["top_score"]
    automatic_label = raw_label if score >= EMOTION_ACCEPT_SCORE and raw_label not in {"unknown", "other"} else "unknown"
    if automatic_label == "unknown":
        flags.append("uncertain_acoustic_emotion")
    manual_label = review["emotion"]
    effective_label = manual_label if manual_label is not None else automatic_label
    annotation_definitions = review.get("attribute_definitions", ATTRIBUTE_DEFINITIONS)
    annotation_schema = signature(annotation_definitions)
    row: JsonObject = dict(record)
    row.update({
        "transcript": review["corrected_text"] if review["corrected_text"] is not None else (None if transcript is None else transcript["text"]),
        "original_transcript": None if transcript is None else transcript["text"],
        "transcription_settings": None if transcript is None else asr_settings,
        "automatic_emotion": automatic_label, "raw_automatic_emotion": raw_label,
        "automatic_emotion_score": score, "emotion_scores": {} if emotion is None else emotion["scores"],
        "emotion_settings": None if emotion is None else emotion_settings,
        "emotion_accept_score": EMOTION_ACCEPT_SCORE,
        "manual_emotion": manual_label, "effective_emotion": effective_label,
        "emotion_source": "human" if manual_label is not None else "model",
        "identity_status": review["identity_status"], "attributes": review["attributes"],
        "attribute_definitions": annotation_definitions, "annotation_schema": annotation_schema,
        "notes": review["notes"], "quality_flags": sorted(set(flags)),
        "analysis_complete": transcript is not None and emotion is not None,
        "extraction_settings": manifest["settings"],
    })
    analysis_rows.append(row)

with (bundle_root / "analysis.jsonl").open("w", encoding="utf-8") as stream:
    for row in analysis_rows:
        stream.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + "\n")
flat_rows: list[JsonObject] = []
for row in analysis_rows:
    flat: JsonObject = {name: row[name] for name in ["observation_id", "actor_id", "character_id", "series_id", "episode_id",
                                        "start_seconds", "end_seconds", "speech_seconds", "transcript",
                                        "automatic_emotion", "automatic_emotion_score", "manual_emotion",
                                        "effective_emotion", "emotion_source", "identity_status", "analysis_complete"]}
    flat.update({f"attribute_{name}": value for name, value in row["attributes"].items()})
    flat["quality_flags"] = ";".join(row["quality_flags"])
    flat_rows.append(flat)
pd.DataFrame(flat_rows).to_csv(bundle_root / "analysis.csv", index=False)
manifest.update({"kind": "voice_stat_analysis", "analysis_created_utc": datetime.now(timezone.utc).isoformat(),
                 "analysis_file": "analysis.jsonl", "attribute_definitions": ATTRIBUTE_DEFINITIONS,
                 "analysis_settings": {"asr": asr_settings, "emotion": emotion_settings,
                                       "emotion_accept_score": EMOTION_ACCEPT_SCORE, "asr_low_logprob": ASR_LOW_LOGPROB},
                 "execution_settings": {"asr_workers": ASR_WORKERS},
                 "analysis_package_versions": {
                     name: importlib.metadata.version(name)
                     for name in ["numpy", "torch", "faster-whisper", "ctranslate2", "funasr", "soundfile"]
                 },
                 "completed_segments": sum(row["analysis_complete"] for row in analysis_rows)})
write_json(bundle_root / "manifest.json", manifest)

archive_path: Path = work_root / f"{manifest['series_id']}_{manifest['episode_id']}_{manifest['character_id']}_analysis.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(bundle_root.rglob("*")):
        if path.is_file() and path.suffix != ".tmp":
            archive.write(path, arcname=str(Path("analysis") / path.relative_to(bundle_root)))
print(f"{manifest['completed_segments']}/{len(records)} clips analyzed. Reviews and caches saved.")
display(pd.DataFrame(flat_rows)[["observation_id", "effective_emotion", "emotion_source", "identity_status", "analysis_complete"]])
files.download(str(archive_path))